In [1]:
import astromugs as mugs
import astromugs.pipeline as pipeline
import astromugs.plotting.plot as mplt

In [2]:
import radmc3dPy as r3d

Fast (Fortran90) Mie-scattering module could not be imported. Falling back to the slower Python version.


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import glob
import re
import os
import time
import subprocess
from contextlib import contextmanager
from matplotlib.collections import PolyCollection

In [5]:
import mugs.casa.pipeline as cas

In [ ]:
# TARGET MOLECULES FOR EMISSION LINES
MOLEC = ["CO","CN","CS","HCN","HCO+","N2H+","CH3OH","H2CO"]  
# PATHS
chemistry_path = Path("/home/fmeyer/Stage/Model23/chemistry_correct/")
thermal_path ="/home/fmeyer/Stage/Model23/thermal/"

In [ ]:
# ==============================================================================
# TARGET PARAMETERS: THE FLYING SAUCER DISK
# Source: Dutrey et al. (2025) - Edge-On Disk Study (EODS) III
# ==============================================================================

flying_saucer_params = {
    # Astrometric position registered to Epoch 2016.0
    'coord': '16h28m13.6979s -24d31m39.491s', 
    
    # Distance of the source located in the Ophiuchus cloud
    'dpc': 120., 
    
    # Systemic velocity in the LSR (Local Standard of Rest) frame
    'v_lsr': 3.72, # [km/s]
    
    # Keplerian velocity normalized at a radius of 100 au
    'v_100au': 2.32, # [km/s]
    
    # Mass of the central Class II T Tauri young stellar object
    'm_star': 0.60, # [Msun]
    
    # Disk geometry: nearly edge-on but slightly inclined
    'inclination': 87.0, # [deg] (Inclination > 85 deg produces strong tomographic projection)
    
    # Position Angle (orientation) of the disk major axis
    'pa': -3.1, # [deg]
}

# ==============================================================================
# DATA FROM ALMA ARCHIVE 2023.1.00907.S
# https://almascience.nrao.edu/aq/?observationsSortProp=project.code&observationsSortDir=asc&observationsProjectCode=2023.1.00907.S&observationsFrequency=339.483..354.618
# ==============================================================================
alma_observations = [
    {
        "antennalist": "alma.cycle10.5.cfg",
        "totaltime": "6985s",
        "refdate": "2024/06/10",
        "pwv": 0.897,
        "integration": "6.048s",
    },
    {
        "antennalist": "alma.cycle10.3.cfg",
        "totaltime": "2298s",
        "refdate": "2024/09/16",
        "pwv": 0.760,
        "integration": "6.048s",
    },
]

In [ ]:
# WRITING INPUT FILES
cas.writing_input_files(chemistry_path,thermal_path,MOLEC,flying_saucer_params,verbose=False)

In [ ]:
# # PLOTTING MOLECULAR DENSITIES (FROM numberdens_XX.inp)
# mplt.numberdens(species=MOLEC, path=thermal_path, vmin=1e-5, vmax=1e7, cmap='jet',
#                ncols=3, xlim=None, ylim=(-100,100), figsize=None,
#                save=False, savename='numberdens.pdf')

In [ ]:
# # PLOTTING GAS PROPERTIES
# mplt.plot_velocity_and_temperature(path=thermal_path, 
#                                   vmin=1, vmax=5, logscale=False, cmap_v='jet',
#                                   Tmin=10, Tmax=100, logscale_T=False, cmap_T='hot',
#                                   xlim=None, ylim=None, figsize=None,
#                                   save=False, savename='gas_properties.pdf')

In [ ]:
# RUNNING radmc3d
# For a first use, it is strongly recommended to use the interactive mode to better understand the function
image_name,widthkms = cas.run_pipeline(
                        available_molecules=MOLEC,
                        interactive=False,
                        alma_bands=[7],
                        molecule="CO",
                        mode="cube",                # "map" for 1 wavelength (needs vkms), "cube" for several ones (needs widthkms and nlam)
                        widthkms=2,
                        nlam=11,
                        freq=345,
                        tol=2,
                        thermal_path=thermal_path,
                        params_dict=flying_saucer_params,
                        lbd=None,
                        lbdrange=None
                    )
# To apply the CASA pipeline to an existing image, comment out the function above and uncomment the line below.
# image_name = 'image_npix_200_incl_87.0_posang_-3.1_sizeau_600_iline_3_CO_widthkms_2_linenlam_11'


In [ ]:
# LOADING AND PLOTTING IMAGE
im = cas.load_and_plot_image(thermal_path,image_name, flying_saucer_params,plot=False,cmap='jet',log=False,arcsec=False,save_png=True)

In [ ]:
# DEFINING FOLDER PATH
base_path = Path("/home/fmeyer/Stage")
image_folder = base_path / image_name

In [ ]:
# WRITING FITS IMAGE FOR CASA
cas.write_fits(image_folder, image_name, flying_saucer_params, im, interactive=True, casa=True,verbose=False)

In [ ]:
# RUNNING CASA PIPELINE TO SIMULATE REAL OBSERVATIONS
cas.casa_pipeline(
    Path("/home/fmeyer/Stage/"),
    image_name,
    flying_saucer_params,
    alma_observations,
    im,  
    widthkms=None,
    verbose=True)

In [ ]:
# PLOTTING CASA OUTPUTS
cas.plot_spectral_cube(
    image_folder,
    filename="image_final.fits",
    cmap="jet",
    figsize=(8, 7),
    zoom_window=(200,300,200,300),
    channel_range=None,
    show_beam=True,
    beam_xy=(215, 215),
    output_prefix="image_plot",
    save_png=True,
    interactive_show=True,
    verbose=False,
)